# Silver Layer — Clean & Conform Hospitality Data
Dedupe, cast types. Keeps booking label is_cancelled; FE drops post-stay leakage.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_date, row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Clean properties
df_p = spark.read.format('delta').table('bronze_properties')
w = Window.partitionBy('property_id').orderBy(col('ingestion_timestamp').desc())
df_p = (
    df_p.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('star_rating', col('star_rating').cast('int'))
    .withColumn('room_count', col('room_count').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_p.write.mode('overwrite').format('delta').saveAsTable('silver_properties')
print(f'silver_properties: {df_p.count()} rows')

In [ ]:
# Clean guests
df_g = spark.read.format('delta').table('bronze_guests')
w2 = Window.partitionBy('guest_id').orderBy(col('ingestion_timestamp').desc())
df_g = (
    df_g.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('total_stays', col('total_stays').cast('int'))
    .withColumn('total_spend', col('total_spend').cast('double'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_g.write.mode('overwrite').format('delta').saveAsTable('silver_guests')
print(f'silver_guests: {df_g.count()} rows')

In [ ]:
# Clean bookings (keep label is_cancelled)
df_b = spark.read.format('delta').table('bronze_bookings')
w3 = Window.partitionBy('booking_id').orderBy(col('ingestion_timestamp').desc())
df_b = (
    df_b.withColumn('_rn', row_number().over(w3)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('check_in_date', to_date('check_in_date'))
    .withColumn('check_out_date', to_date('check_out_date'))
    .withColumn('nights', col('nights').cast('int'))
    .withColumn('room_rate', col('room_rate').cast('double'))
    .withColumn('lead_time_days', col('lead_time_days').cast('int'))
    .withColumn('is_refundable', col('is_refundable').cast('int'))
    .withColumn('is_cancelled', col('is_cancelled').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_b.write.mode('overwrite').format('delta').saveAsTable('silver_bookings')
print(f'silver_bookings: {df_b.count()} rows')

In [ ]:
# Clean reviews
df_r = spark.read.format('delta').table('bronze_reviews')
df_r = (
    df_r
    .withColumn('review_date', to_date('review_date'))
    .withColumn('overall_score', col('overall_score').cast('double'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_r.write.mode('overwrite').format('delta').saveAsTable('silver_reviews')
print(f'silver_reviews: {df_r.count()} rows')